# Proyecto No. 2 - Data Science

**Integrantes:** Jose Ordoñez, José Antón, Adrián Gonzáles, Gadiel Ocaña

**Dataset:** Predicting Effective Arguments (https://www.kaggle.com/competitions/feedback-prize-effectiveness)

## Situación problemática

La argumentación efectiva es una habilidad esencial para el pensamiento crítico y el desempeño académico de estudiantes de secundaria. Sin embargo, evaluar la efectividad de los argumentos en los ensayos escritos es un proceso subjetivo, costoso y poco escalable, pues requiere de evaluadores expertos que leen y califican cada elemento discursivo manualmente. Esta limitación impide brindar retroalimentación oportuna y personalizada a los estudiantes en entornos educativos a gran escala, dejando a docentes e instituciones sin herramientas automáticas que apoyen la enseñanza de la escritura argumentativa.

## Problema Científico

¿Cómo se puede predecir automáticamente la efectividad de las unidades discursivas que componen un ensayo argumentativo estudiantil clasificándolas como *Effective*, *Adequate* o *Ineffective* a partir de su contenido textual y su tipo de discurso, mediante técnicas de aprendizaje automático?

## Objetivos

**Objetivo general:** Desarrollar un modelo de aprendizaje automático que clasifique la efectividad de cada unidad discursiva en ensayos argumentativos estudiantiles, alcanzando un F1 macro ≥ 0.55 en el conjunto de prueba.

**Objetivos específicos:**
1. Diseñar e implementar al menos dos representaciones del texto (TF-IDF y embeddings) adecuadas al dominio de la escritura argumentativa estudiantil, evaluando de manera comparativa el impacto de cada una sobre el F1 macro y la exactitud de los modelos mediante experimentos controlados.

2. Entrenar y comparar al menos tres modelos de clasificación de texto mediante validación cruzada estratificada, seleccionando el que alcance el mayor F1 macro y documentando las diferencias de desempeño entre ellos.

## Descripción de los datos

Cargamos el train.csv y el test.csv que provee la competencia. Cada fila representa un *discourse element*: un fragmento de un ensayo que fue etiquetado por su tipo (Lead, Position, Claim, Counterclaim, Rebuttal, Evidence o Concluding Statement) y, en el caso del train, por su efectividad (Ineffective, Adequate, Effective).

In [1]:
import pandas as pd

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print('train:', train.shape)
print('test:', test.shape)
train.head()

train: (36765, 5)
test: (10, 4)


,discourse_id,essay_id,discourse_text,discourse_type,discourse_effectiveness
0,0013cc385424,007ACE74B050,"Hi, i'm Isaac, i'm going to be writing about h...",Lead,Adequate
1,9704a709b505,007ACE74B050,"On my perspective, I think that the face is a ...",Position,Adequate
2,c22adee811b6,007ACE74B050,I think that the face is a natural landform be...,Claim,Adequate
3,a10d361e54e4,007ACE74B050,"If life was on Mars, we would know by now. The...",Evidence,Adequate
4,db3e453ec4e2,007ACE74B050,People thought that the face was formed by ali...,Counterclaim,Adequate


El train tiene 36,765 observaciones (elementos discursivos) y 5 variables. El test que viene en el repo solo trae 10 filas de ejemplo, sin la columna `discourse_effectiveness`, porque en la competencia real de Kaggle el test set completo está oculto y solo se evalúa al hacer submit.

In [2]:
train.dtypes

discourse_id               object
essay_id                   object
discourse_text             object
discourse_type             object
discourse_effectiveness    object
dtype: object

Las 5 variables son:

| Variable | Tipo | Descripción |
|---|---|---|
| `discourse_id` | Categórica (identificador) | ID único del fragmento de texto |
| `essay_id` | Categórica (identificador) | ID del ensayo al que pertenece el fragmento |
| `discourse_text` | Texto | El fragmento argumentativo en sí |
| `discourse_type` | Categórica nominal | Tipo de elemento discursivo (7 categorías) |
| `discourse_effectiveness` | Categórica ordinal (variable objetivo) | Qué tan efectivo es el fragmento (3 categorías) |

In [3]:
train.isnull().sum()

discourse_id               0
essay_id                   0
discourse_text             0
discourse_type             0
discourse_effectiveness    0
dtype: int64

In [4]:
print('IDs de discurso duplicados:', train['discourse_id'].duplicated().sum())
print('Filas completamente duplicadas:', train.duplicated().sum())

IDs de discurso duplicados: 0
Filas completamente duplicadas: 0


No hay valores nulos ni filas duplicadas. Es un dataset bastante limpio en ese sentido.

In [5]:
print('Ensayos únicos en train:', train['essay_id'].nunique())
train.groupby('essay_id').size().describe()

Ensayos únicos en train: 4191


count    4191.000000
mean        8.772369
std         3.492605
min         1.000000
25%         7.000000
50%         9.000000
75%        11.000000
max        23.000000
dtype: float64

Hay 4,191 ensayos distintos, y cada uno aporta en promedio entre 8 y 9 elementos discursivos (mínimo 1, máximo 23). Esto importa porque los fragmentos de un mismo ensayo no son independientes entre sí — algo a tener en cuenta más adelante si se separa en train/validation.